<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/intake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Intake, screens and micro sieves***

In [ ]:
import pandas as pd
import math as mh

def intake(Partable2, dailyflow, Treatment,lanes):

    """
    Calculate the dimensions and parameters for an inlet screen in a water treatment plant.

    Parameters:
    - flow_rate (float): Total flow rate through the plant in m³/s.
    - screen_vel (float): Desired velocity of water approaching the screen in m/s.
    - bar_spacing (float): Spacing between the bars in meters.
    - bar_width (float): Thickness of the bars in meters.
    - screen_width (float): Width of the screen in meters.
    - blockage_factor (float): Fraction of the screen area blocked by bars (0-1, e.g., 0.15 for 15%).

    Returns:
    - results: A dictionary containing the calculated screen parameters.
    """
    flowlane=dailyflow/lanes
    flowrate=flowlane*0.000012 # m3/day to m3/s

    # Calculate the gross screen area
    screen_vel=float(input("Desired velocity of water approaching the screen (m/s):"))
    screen_area_lane = mh.ceil(flowrate / screen_vel)
    screen_area_total = screen_area_lane * lanes
    # Calculate the net open area of the screen
    blockage_factor=float(input("Blockage factor (0-1):"))
    bar_spacing=float(input("Bar spacing (m):"))
    bar_width=float(input("Bar width (m):"))
    open_area_ratio = (bar_spacing / (bar_spacing + bar_width)) * (1 - blockage_factor)
    net_open_area = round(screen_area_lane * open_area_ratio,2)

    # Calculate the velocity through the bars
    velocity_through_bars = flowrate / net_open_area

    # Head loss coefficient based on bar shape (default to 1.8 for typical bars)
    head_loss_coefficient = 1.8

    # Calculate the head loss using the Bernoulli principle
    gravitational_acceleration = 9.81  # m/s²
    screen_headloss = head_loss_coefficient * (velocity_through_bars ** 2) / (2 * gravitational_acceleration)
    screen_headloss=mh.ceil(screen_headloss)
    # Compile results
    print("Gross Area per lane (m²):", screen_area_lane)
    print("Total Gross Area (m²):", screen_area_total)
    print("Net Open Area per lane (m²):", net_open_area)
    print("Velocity Through Bars at each lane (m/s):", velocity_through_bars)
    print("Head Loss at each lane (m):", screen_headloss)
    print("Bar Spacing (m):", bar_spacing)
    print("Blockage Factor (%):", blockage_factor * 100)

    """
    Calculate the net area of a micro sieve considering the open area ratio.

    Parameters:
    - flowrate (float): Flow rate of water (m³/s).
    - flow_velocity (float): Velocity of water through the sieve (m/s).
    - efficiency_factor (float): Efficiency factor of the sieve (default: 0.8).
    - open_area_ratio (float): Open area ratio of the sieve (default: 0.5).

    Returns:
    - tuple: (gross_area, net_area) where:
        - microsieves_area (float): Total area of the sieve (m²).
        - microsieves_net_area (float): Open (net) area of the sieve (m²).
    """

    if (Treatment['Treatment Steps'] == "Microsieves").any():

        flow_velocity = float(input("Velocity of water through the sieve (m/s):"))
        efficiency_factor = 0.8
        open_area_ratio = 0.5
        if flow_velocity <= 0:
           raise ValueError("Flow velocity must be greater than zero.")
        if not (0 < efficiency_factor <= 1):
           raise ValueError("Efficiency factor must be between 0 and 1.")
        if not (0 < open_area_ratio <= 1):
           raise ValueError("Open area ratio must be between 0 and 1.")

    # Calculate gross area
        microsieves_area_lane = mh.ceil((flowrate / (flow_velocity * efficiency_factor)))
        microsieves_area=round(lanes*microsieves_area_lane)
    # Calculate net area
        microsieves_net_area = microsieves_area * open_area_ratio

        print(f"Gross area of the micro sieve per lane: {microsieves_area_lane:.2f} m²")
        print(f"Net area of the lane micro sieve: {microsieves_net_area:.2f} m²")
        print(f"Gross area of the micro sieves: {microsieves_area:.2f} m²")

   # Calculate the adjusted velocity through the open area
        velocity_open = flow_velocity / open_area_ratio

    # Calculate head loss
        head_loss_coefficient=1.2
        gravity=9.81
        microsieves_headloss = head_loss_coefficient * (velocity_open**2) / (2 * gravity)

    else:
        microsieves_headloss=0
        microsieves_area=0

    return screen_headloss, screen_area_total,microsieves_headloss,microsieves_area

In [ ]:
  def pumps(Partable2, dailyflow, Treatment, flowlane, lanes):

    """
    Function to calculate pump characteristics and pumping station requirements.
    """
    print("")
    print("SURFACE WATER INTAKE AND SCREENS")
    flowrate=max(flowlane)*0.000012 # m3/day to m3/s
    # Step 1: Ask if the intake pumping station is required
    response = input("Is an intake pumping station required for the drinking water treatment plant? (Yes/No): ").strip().lower()
    if response== 'No':
        print("No intake pumping station is required.")
        Pump_power=0
        Pump_area=0
    else:
        # Step 2: Gather user inputs for calculations
        pipe_length = float(input("Intake pipe length estimation (in meters): "))
        pipe_diameter = float(input("Intake pipe diameter estimation (in meters): "))
        pipe_roughness = float(input("Enter the pipe roughness coefficient (0.03 - 0.15 for typical pipes): "))
        level_difference = float(input("Level difference between source and intake tank (in meters): "))
        pump_efficiency = float(input("Pump efficiency estimation (in percentage, e.g., 80 for 80%): "))
        motor_efficiency = float(input("Motor efficiency estimation (in percentage, e.g., 90 for 90%): "))
        num_pumps = int(input("Enter the number of pumps to be installed (excluding stand-by pumps): "))
        if flowrate< 1:
            area_per_pump = 5
        else:
            area_per_pump =flowrate*5
            print("Estimation of 5 m2 required per m3/s of flow capacity with 5m2 minimum size")

    # Step 3: Calculate the head
        velocity = (4 * flowrate) / (mh.pi * pipe_diameter**2)  # Velocity in m/s
        reynolds_number = (velocity * pipe_diameter) / (1e-6)  # Assuming water kinematic viscosity is 1e-6 m^2/s
        friction_factor = 0.25 / (mh.log10(pipe_roughness / (3.7 * pipe_diameter) + 5.74 / reynolds_number**0.9)**2)  # Darcy-Weisbach equation
        head_loss = (friction_factor * pipe_length * velocity**2) / (2 * 9.81 * pipe_diameter)  # Head loss in meters
        total_head = 1.15*(level_difference + head_loss)  # Total head in meters

    # Step 4: Calculate power consumption and other characteristics
        pump_efficiency /= 100  # Convert efficiency to decimal
        motor_efficiency /= 100  # Convert efficiency to decimal

    # Power calculation: Power (kW) = (Flow (m^3/s) * Head (m) * 9.81) / (Pump Eff. * Motor Eff.)
        power_per_pump = (flowrate/num_pumps * total_head * 9.81) / (pump_efficiency * motor_efficiency)  # kW
        lane_total_power = power_per_pump * num_pumps
        Pump_power = round(lane_total_power * lanes)
    # Step 5: Calculate the total area required for the pumping station
        Pump_area = round(lanes*(num_pumps * area_per_pump))

    # Step 6: Display results
        print("\nPumping Station Characteristics:")
        print(f"  - Flow Rate: {flowrate:.2f} m^3/s")
        print(f"  - Total Head: {total_head:.2f} meters")
        print(f"  - Pump Efficiency: {pump_efficiency * 100:.2f}%")
        print(f"  - Motor Efficiency: {motor_efficiency * 100:.2f}%")
        print(f"  - Number of Pumps (including stand-by): {num_pumps+1}")
        print(f"  - Lane Pumping Station Power Consumption: {power_per_pump:.2f} kW")
        print(f"  - Total Lane Power Consumption: {lane_total_power:.2f} kW")
        print(f"  - Total Power Consumption: {Pump_power:.2f} kW")
        print(f"  - Total Pumping Station Area: {Pump_area:.2f} square meters")

      return Pump_power, Pump_area